<a href="https://colab.research.google.com/github/BringzUU/lab-ai-69/blob/main/NakhonPathom_recommender_By_FANTA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ระบบแนะนำที่เที่ยวในตัวเมืองนครปฐม
Content-based Recommendation (TF-IDF + Cosine Similarity)

**วิธีใช้:** กด Runtime → Run all แล้วเลื่อนลงล่างสุดเพื่อใช้งานแบบกดเลือก

*ข้อมูลเป็นตัวอย่างเริ่มต้น ควรตรวจสอบและเพิ่มสถานที่/คะแนนจริงก่อนส่งงาน*

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# price: 0=ฟรี, 1=ไม่เกิน 100 บาท, 2=มากกว่า 100 บาท | rating = ค่าตัวอย่าง 1-5
DATA = [
    ("พระปฐมเจดีย์", "วัด", "ไหว้พระ ทำบุญ สถาปัตยกรรม ถ่ายรูป ประวัติศาสตร์", 0, "เช้า เย็น", 4.8,
     "เจดีย์ใหญ่ ศูนย์กลางของเมืองนครปฐม"),
    ("พระราชวังสนามจันทร์", "ประวัติศาสตร์", "วัง สถาปัตยกรรม ถ่ายรูป สวน ประวัติศาสตร์ เดินเล่น", 1, "เช้า บ่าย", 4.6,
     "พระราชวังและสวนร่มรื่น เหมาะเดินถ่ายรูป"),
    ("พิพิธภัณฑสถานแห่งชาติ พระปฐมเจดีย์", "พิพิธภัณฑ์", "พิพิธภัณฑ์ โบราณคดี ทวารวดี ประวัติศาสตร์ เรียนรู้", 1, "เช้า บ่าย", 4.2,
     "จัดแสดงโบราณวัตถุสมัยทวารวดี"),
    ("วัดพระประโทณเจดีย์", "วัด", "ไหว้พระ โบราณสถาน ทวารวดี ประวัติศาสตร์ สงบ", 0, "เช้า เย็น", 4.1,
     "วัดและโบราณสถานเก่าแก่"),
    ("วัดพระงาม", "วัด", "ไหว้พระ ทำบุญ สงบ ประวัติศาสตร์", 0, "เช้า เย็น", 4.0,
     "วัดเก่าแก่ในเมือง บรรยากาศสงบ"),
    ("ตลาดโต้รุ่งหน้าพระปฐมเจดีย์", "ตลาด", "ของกิน อาหาร ตลาด กลางคืน ของฝาก เดินเล่น", 1, "เย็น กลางคืน", 4.4,
     "ตลาดอาหารและของกินใกล้พระปฐมเจดีย์"),
    ("สวนสาธารณะหนองสาหร่าย", "สวนสาธารณะ", "ออกกำลังกาย ธรรมชาติ ปั่นจักรยาน พักผ่อน เดินเล่น", 0, "เช้า เย็น", 4.0,
     "สวนสาธารณะสำหรับวิ่ง เดิน พักผ่อน"),
    ("มหาวิทยาลัยศิลปากร วังสนามจันทร์", "แหล่งเรียนรู้", "สถาปัตยกรรม ถ่ายรูป เดินเล่น ประวัติศาสตร์ ศิลปะ", 0, "เช้า บ่าย", 4.3,
     "บรรยากาศวังเก่าในรั้วมหาวิทยาลัย"),
    ("คาเฟ่ในตัวเมือง (ตัวอย่าง)", "คาเฟ่", "คาเฟ่ กาแฟ ขนม ถ่ายรูป พักผ่อน", 2, "บ่าย เย็น", 4.2,
     "แทนที่ด้วยชื่อคาเฟ่จริงที่คุณสำรวจ"),
    ("ร้านอาหารพื้นเมือง (ตัวอย่าง)", "ร้านอาหาร", "อาหาร ของกิน อาหารไทย ร้านดัง", 2, "เช้า บ่าย เย็น", 4.1,
     "แทนที่ด้วยร้านอาหารจริงในเมือง"),
]
COLS = ["name", "category", "tags", "price", "best_time", "rating", "desc"]
PRICE_LABEL = {0: "ฟรี", 1: "ไม่เกิน 100 บาท", 2: "มากกว่า 100 บาท"}

df = pd.DataFrame(DATA, columns=COLS)
docs = df["category"] + " " + df["tags"]
vec = TfidfVectorizer(token_pattern=r"[^\s]+")   # แท็กคั่นด้วยช่องว่าง ไม่ต้องตัดคำไทย
matrix = vec.fit_transform(docs)
all_tags = sorted({t for tags in df["tags"] for t in tags.split()})
print("จำนวนสถานที่:", len(df), "| จำนวนแท็ก:", len(all_tags))

จำนวนสถานที่: 10 | จำนวนแท็ก: 29


In [2]:
def _rank(sim, max_price, time_of_day, k, exclude=None):
    out = df.copy()
    out["score"] = (0.8 * sim + 0.2 * (out["rating"] / 5)).round(3)
    out = out[out["price"] <= max_price]
    if time_of_day != "ไม่ระบุ":
        out = out[out["best_time"].str.contains(time_of_day)]
    if exclude:
        out = out[out["name"] != exclude]
    out = out.sort_values("score", ascending=False).head(k)
    out["ค่าใช้จ่าย"] = out["price"].map(PRICE_LABEL)
    return out[["name", "category", "ค่าใช้จ่าย", "best_time", "score", "desc"]].reset_index(drop=True)


def recommend_by_interest(picked, max_price=2, time_of_day="ไม่ระบุ", k=5):
    """แนะนำตามความสนใจ (picked = list ของแท็ก)"""
    if picked:
        sim = cosine_similarity(vec.transform([" ".join(picked)]), matrix).ravel()
    else:
        sim = 0 * df["rating"].values
    return _rank(sim, max_price, time_of_day, k)


def recommend_similar(liked, max_price=2, time_of_day="ไม่ระบุ", k=5):
    """แนะนำสถานที่ที่คล้ายกับที่ชอบ"""
    i = df.index[df["name"] == liked][0]
    sim = cosine_similarity(matrix[i], matrix).ravel()
    return _rank(sim, max_price, time_of_day, k, exclude=liked)


# ตัวอย่างการใช้งาน
print("== ชอบถ่ายรูป + ประวัติศาสตร์ ==")
display(recommend_by_interest(["ถ่ายรูป", "ประวัติศาสตร์"], k=4))
print("== คล้าย 'พระปฐมเจดีย์' ==")
display(recommend_similar("พระปฐมเจดีย์", k=4))

== ชอบถ่ายรูป + ประวัติศาสตร์ ==


,name,category,ค่าใช้จ่าย,best_time,score,desc
0,พระราชวังสนามจันทร์,ประวัติศาสตร์,ไม่เกิน 100 บาท,เช้า บ่าย,0.629,พระราชวังและสวนร่มรื่น เหมาะเดินถ่ายรูป
1,พระปฐมเจดีย์,วัด,ฟรี,เช้า เย็น,0.579,เจดีย์ใหญ่ ศูนย์กลางของเมืองนครปฐม
2,มหาวิทยาลัยศิลปากร วังสนามจันทร์,แหล่งเรียนรู้,ฟรี,เช้า บ่าย,0.525,บรรยากาศวังเก่าในรั้วมหาวิทยาลัย
3,วัดพระงาม,วัด,ฟรี,เช้า เย็น,0.321,วัดเก่าแก่ในเมือง บรรยากาศสงบ


== คล้าย 'พระปฐมเจดีย์' ==


,name,category,ค่าใช้จ่าย,best_time,score,desc
0,วัดพระงาม,วัด,ฟรี,เช้า เย็น,0.730,วัดเก่าแก่ในเมือง บรรยากาศสงบ
1,พระราชวังสนามจันทร์,ประวัติศาสตร์,ไม่เกิน 100 บาท,เช้า บ่าย,0.516,พระราชวังและสวนร่มรื่น เหมาะเดินถ่ายรูป
2,วัดพระประโทณเจดีย์,วัด,ฟรี,เช้า เย็น,0.487,วัดและโบราณสถานเก่าแก่
3,มหาวิทยาลัยศิลปากร วังสนามจันทร์,แหล่งเรียนรู้,ฟรี,เช้า บ่าย,0.473,บรรยากาศวังเก่าในรั้วมหาวิทยาลัย


## ใช้งานแบบกดเลือก

In [3]:
import ipywidgets as w
from IPython.display import display, clear_output

tag_box = w.SelectMultiple(options=all_tags, rows=8, description="สนใจ", layout=w.Layout(width="400px"))
place_dd = w.Dropdown(options=list(df["name"]), description="ชอบที่", layout=w.Layout(width="500px"))
price_dd = w.Dropdown(options=[("ฟรีเท่านั้น", 0), ("ไม่เกิน 100 บาท", 1), ("ไม่จำกัด", 2)], value=2, description="งบ")
time_dd = w.Dropdown(options=["ไม่ระบุ", "เช้า", "บ่าย", "เย็น", "กลางคืน"], description="ช่วงเวลา")
k_sl = w.IntSlider(value=5, min=1, max=len(df), description="จำนวน")
btn1 = w.Button(description="แนะนำตามความสนใจ", button_style="primary", layout=w.Layout(width="200px"))
btn2 = w.Button(description="หาที่คล้ายที่ชอบ", button_style="success", layout=w.Layout(width="200px"))
out = w.Output()

def on_interest(_):
    with out:
        clear_output()
        display(recommend_by_interest(list(tag_box.value), price_dd.value, time_dd.value, k_sl.value))

def on_similar(_):
    with out:
        clear_output()
        display(recommend_similar(place_dd.value, price_dd.value, time_dd.value, k_sl.value))

btn1.on_click(on_interest)
btn2.on_click(on_similar)
display(w.VBox([tag_box, place_dd, price_dd, time_dd, k_sl, w.HBox([btn1, btn2]), out]))